# nitid 2: Training and evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nitidio/nitid/blob/main/examples/notebooks/02_training.ipynb)

This notebook fine-tunes a pretrained detection model on a dataset, validates
it and exports it:

1. Prepare a dataset (COCO-mini, downloaded on first use)
2. Fine-tune `model1s`
3. Validate and look at the plots
4. Predict with the trained model
5. Export it

Start with [1: Inference](01_inference.ipynb) if you have not used nitid yet.

## Setup

**Google Colab:** select a GPU runtime (*Runtime → Change runtime type*). The
cell below installs `nitid[train]` from PyPI.

**Local:** from a nitid checkout, install the training extra and start Jupyter:

```bash
uv sync --extra train
uv run --with jupyterlab jupyter lab
```

With a GPU the whole notebook takes a few minutes; on a CPU it works, but
takes about 25 minutes. Every output is written under
`runs/notebooks/`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nitid[train]"], check=True)


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "dfine").is_dir():
            return candidate
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ROOT = REPO_ROOT or Path.cwd()
WORK_DIR = ROOT / "runs" / "notebooks"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)  # downloaded checkpoints and datasets/ live here


def repo_file(path: str) -> Path:
    """Return a file from the nitid repository, downloading it outside a checkout."""
    local = ROOT / path
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"https://raw.githubusercontent.com/Nitidio/nitid/main/{path}", local)
    return local


print(f"Colab: {IN_COLAB}")
print(f"nitid checkout: {REPO_ROOT}")
print(f"Outputs go to: {WORK_DIR}")

In [ ]:
import torch

from nitid import NITID

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 1. Dataset

nitid reads three dataset layouts without a conversion step:

- **COCO JSON**: `images/<split>/` plus `annotations/instances_<split>.json`
- **YOLO .txt**: `images/<split>/` plus `labels/<split>/`, or split-first `<split>/images/`
- **Semantic masks**: one single-channel class-ID PNG per image, for `task="semantic"`

A YAML file points at the dataset and lists the classes:

```yaml
path: /data/my_dataset
train: images/train
val: images/val
names:
  0: person
  1: car
```

The [fine-tuning guide](https://github.com/Nitidio/nitid/blob/main/docs/fine_tuning.md) covers every layout, the
category-ID mapping and the semantic-mask options.

This notebook uses **COCO-mini**: 128 train and 32 val images from COCO 2017,
restricted to redistributable licences, with box and mask annotations. It is
meant to check the whole pipeline in minutes, not to reach a good mAP. Its YAML
declares a `download` URL and checksum, so nitid fetches the archive into
`datasets/coco-mini/` on first use.

In [ ]:
DATA_YAML = repo_file("configs/datasets/coco-mini.yml")
print(DATA_YAML.read_text().split("\nnc:")[0])

## 2. Fine-tune

`train()` rebuilds the class head when the dataset's classes differ from the
checkpoint's, keeping the pretrained backbone. Five epochs are enough to see
the pipeline work; a real dataset needs far more (around 100-200).

For instance segmentation, use `NITID("model1s", task="segment")` with the same
dataset: COCO-mini has masks too.

In [ ]:
EPOCHS = 5

model = NITID("model1s", task="detect")
metrics = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=640,
    batch=8,
    device=DEVICE,
    project=WORK_DIR / "train",
    name="coco-mini",
    exist_ok=True,
)
print(f"mAP50: {metrics['mAP50']:.4f}  mAP50-95: {metrics['mAP50-95']:.4f}")

The run directory holds `best.pth` and `last.pth`, self-contained checkpoints
that load with `NITID("path/to/best.pth")`.

The CLI equivalent is:

```bash
nitid train model=model1s task=detect data=configs/datasets/coco-mini.yml epochs=5 batch=8
```

## 3. Validate

`val()` computes COCO metrics on the validation split and saves plots such as
the confusion matrix and the precision-recall curves.

In [ ]:
from IPython.display import Image, display

metrics = model.val(data=str(DATA_YAML), project=WORK_DIR / "val", name="coco-mini", exist_ok=True)
print(f"mAP50: {metrics['mAP50']:.4f}  mAP50-95: {metrics['mAP50-95']:.4f}")

for plot in sorted((WORK_DIR / "val" / "coco-mini").glob("*.png")):
    print(plot.name)
    display(Image(filename=str(plot), width=600))

## 4. Predict with the trained model

After `train()`, `model` holds the fine-tuned weights:

In [ ]:
val_images = sorted((ROOT / "datasets" / "coco-mini" / "images" / "val").glob("*.jpg"))[:3]

for image_path in val_images:
    result = model.predict(str(image_path), conf=0.5)[0]
    out = WORK_DIR / "predict" / image_path.name
    out.parent.mkdir(parents=True, exist_ok=True)
    result.save(str(out))
    print(image_path.name, "-", len(result), "objects")
    display(Image(filename=str(out), width=500))

## 5. Export

Exported graphs include the postprocessor, so they return
`(labels, boxes, scores)`. ONNX:

In [ ]:
onnx_path = model.export(
    format="onnx", output=WORK_DIR / "export" / "coco-mini.onnx", exist_ok=True
)
print(onnx_path)

OpenVINO, for Intel CPU, iGPU and NPU, needs `pip install "nitid[openvino]"`;
the cell skips it when the extra is missing. TensorRT and the other options
are in the [export guide](https://github.com/Nitidio/nitid/blob/main/docs/export.md).

In [ ]:
try:
    import openvino as ov
except ImportError:
    print('OpenVINO is not installed: pip install "nitid[openvino]"')
else:
    xml_path = model.export(
        format="openvino", output=WORK_DIR / "export" / "coco-mini.xml", exist_ok=True
    )
    compiled = ov.Core().compile_model(xml_path, "AUTO")
    print("compiled", xml_path, "for", compiled.get_property("EXECUTION_DEVICES"))

## Next

- Train on your own data: [fine-tuning guide](https://github.com/Nitidio/nitid/blob/main/docs/fine_tuning.md)
- Deploy: [export guide](https://github.com/Nitidio/nitid/blob/main/docs/export.md) and
  [training on CUDA and OpenVINO machines](https://github.com/Nitidio/nitid/blob/main/docs/training_cuda_openvino.md)